In [4]:
import numpy as np
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
import plotly.graph_objects as go

# ============================================================
# Pairwise Net Spillover + SOLVPN Outgoing Spillover
# ============================================================
# GFEVD definition:
# theta[t, i, j]
# row i = response variable
# col j = shock source
#
# theta[target, source] = source -> target spillover
#
# Pairwise net:
# pairwise_net[source, target]
# = theta[target, source] - theta[source, target]
#
# > 0 : source -> target 우세
# < 0 : target -> source 우세
# ============================================================

# ============================================================
# Config
# ============================================================

BASE_DIR = Path("./")
OUT_DIR = BASE_DIR / "pairwise_output"
OUT_DIR.mkdir(parents=True, exist_ok=True)

GFEVD_FILE = BASE_DIR / "gfevd_all.npy"

DATE_FILE_CANDIDATES = [
    BASE_DIR / "tvpvar_input_scaled.csv",
    BASE_DIR / "tvpvar_input_transformed.csv",
    BASE_DIR / "tvpvar_raw_level_merged.csv",
]

DEFAULT_VAR_NAMES = [
    "dlog_SOLVPN",
    "dlog_COPPER",
    "dlog_GOLD",
    "dlog_SILVER",
    "dlog_DXY",
    "dlog_OIL",
    "dlog_GAS",
    "d_UST10Y",
    "d_VIX",
]

SOLVPN_NAME = "dlog_SOLVPN"

SELECTED_PAIRS = [
    ("dlog_SOLVPN", "dlog_COPPER"),
    ("dlog_COPPER", "dlog_SOLVPN"),
    ("dlog_SOLVPN", "dlog_GOLD"),
    ("dlog_SOLVPN", "dlog_SILVER"),
    ("dlog_SOLVPN", "d_VIX"),
]

SHOW_INTERACTIVE = True
SAVE_SELECTED_PAIR_PNG = True

EPS = 1e-12


# ============================================================
# Helper functions
# ============================================================

def infer_gfevd_tnn(arr: np.ndarray) -> np.ndarray:
    """
    Convert GFEVD array to shape (T, N, N).
    Accepts:
    - (T, N, N)
    - (N, N, T)
    """
    if arr.ndim != 3:
        raise ValueError(f"gfevd_all must be 3D. Got shape={arr.shape}")

    # Expected: (T, N, N)
    if arr.shape[1] == arr.shape[2]:
        return arr.astype(float, copy=True)

    # Alternative: (N, N, T)
    if arr.shape[0] == arr.shape[1]:
        return np.transpose(arr, (2, 0, 1)).astype(float, copy=True)

    raise ValueError(f"Cannot infer GFEVD shape. Got shape={arr.shape}")


def infer_var_names(n: int) -> list:
    """
    Use DEFAULT_VAR_NAMES if length matches N.
    Otherwise generate generic variable names.
    """
    if len(DEFAULT_VAR_NAMES) == n:
        return DEFAULT_VAR_NAMES

    print(f"[WARN] DEFAULT_VAR_NAMES length={len(DEFAULT_VAR_NAMES)} != N={n}")
    print("[WARN] Generic variable names will be used.")
    return [f"VAR{i+1}" for i in range(n)]


def load_dates(target_len: int):
    """
    Load Date column from candidate files.
    Align by taking last target_len rows.
    """
    for fp in DATE_FILE_CANDIDATES:
        if not fp.exists():
            continue

        try:
            df = pd.read_csv(fp)

            date_col = None
            for c in ["Date", "date", "DATE"]:
                if c in df.columns:
                    date_col = c
                    break

            if date_col is None:
                continue

            dates = pd.to_datetime(df[date_col], errors="coerce")
            dates = dates.dropna().reset_index(drop=True)

            if len(dates) < target_len:
                print(f"[WARN] {fp} has {len(dates)} valid dates, but target_len={target_len}. Skipped.")
                continue

            return dates.iloc[-target_len:].reset_index(drop=True)

        except Exception as e:
            print(f"[WARN] Failed to read dates from {fp}: {e}")

    return None


def normalize_gfevd_to_percent(gfevd_tnn: np.ndarray) -> np.ndarray:
    """
    Normalize each row of each GFEVD matrix to sum to 100.
    This works whether the input is 0~1 scale or 0~100 scale.
    """
    arr = np.asarray(gfevd_tnn, dtype=float).copy()

    if not np.all(np.isfinite(arr)):
        raise ValueError("GFEVD contains NaN or inf.")

    min_val = arr.min()
    if min_val < -1e-10:
        raise ValueError(f"GFEVD contains negative values. min={min_val}")

    arr[arr < 0] = 0.0

    row_sum = arr.sum(axis=2, keepdims=True)

    if np.any(row_sum <= EPS):
        bad = np.argwhere(row_sum.squeeze(-1) <= EPS)
        raise ValueError(f"Some GFEVD rows have zero row sum. First bad index={bad[0].tolist()}")

    theta_pct = arr / row_sum * 100.0

    max_err = np.max(np.abs(theta_pct.sum(axis=2) - 100.0))
    if max_err > 1e-8:
        raise RuntimeError(f"Row normalization failed. max_row_err={max_err}")

    return theta_pct


def compute_pairwise_net(theta_pct: np.ndarray) -> np.ndarray:
    """
    theta_pct[target, source] = source -> target spillover

    pairwise_net[source, target]
    = theta[target, source] - theta[source, target]
    """
    if theta_pct.ndim != 2 or theta_pct.shape[0] != theta_pct.shape[1]:
        raise ValueError(f"theta_pct must be square 2D matrix. Got {theta_pct.shape}")

    return theta_pct.T - theta_pct


def save_heatmap_png(df_mean: pd.DataFrame, out_path: Path):
    """
    Save cold-hot heatmap.
    Row = Source, Column = Target.
    Positive value = row source dominates column target.
    """
    values = df_mean.values
    vmax = np.nanmax(np.abs(values))

    fig, ax = plt.subplots(figsize=(10, 8))

    im = ax.imshow(
        values,
        cmap="coolwarm",
        vmin=-vmax,
        vmax=vmax
    )

    labels = list(df_mean.index)

    ax.set_xticks(np.arange(len(labels)))
    ax.set_yticks(np.arange(len(labels)))
    ax.set_xticklabels(labels, rotation=45, ha="right")
    ax.set_yticklabels(labels)

    for i in range(len(labels)):
        for j in range(len(labels)):
            ax.text(
                j,
                i,
                f"{values[i, j]:.2f}",
                ha="center",
                va="center",
                fontsize=8
            )

    ax.set_title("Mean Pairwise Net Spillover\nRow = Source, Column = Target")
    fig.colorbar(im, ax=ax, label="Pairwise NET (%)")

    plt.tight_layout()
    plt.savefig(out_path, dpi=300, bbox_inches="tight")
    plt.close()


def save_pair_series_png(df_time: pd.DataFrame, x_col: str, pair_col: str, out_path: Path):
    """
    Save static pairwise net series PNG.
    """
    x = pd.to_datetime(df_time[x_col]) if x_col == "Date" else df_time[x_col]

    plt.figure(figsize=(12, 5))
    plt.plot(x, df_time[pair_col], linewidth=1.3)
    plt.axhline(0.0, linestyle="--", linewidth=1.0)

    plt.title(f"Pairwise Net Spillover: {pair_col}")
    plt.xlabel(x_col)
    plt.ylabel("Pairwise NET (%)")

    plt.tight_layout()
    plt.savefig(out_path, dpi=300, bbox_inches="tight")
    plt.close()


def show_pair_series_interactive(df_time: pd.DataFrame, x_col: str, pair_cols: list, title: str):
    """
    Show interactive pairwise net line chart inside notebook/Colab.
    No HTML file is saved.
    """
    fig = go.Figure()

    for col in pair_cols:
        fig.add_trace(
            go.Scatter(
                x=df_time[x_col],
                y=df_time[col],
                mode="lines",
                name=col
            )
        )

    fig.add_hline(y=0.0)

    fig.update_layout(
        title=title,
        xaxis_title=x_col,
        yaxis_title="Pairwise NET (%)",
        height=600,
        hovermode="x unified"
    )

    fig.show()


def show_solvpn_to_all_interactive(df_solvpn_to_all: pd.DataFrame, x_col: str, title: str):
    """
    Show interactive raw directional spillover:
    SOLVPN -> all other variables.
    """
    fig = go.Figure()

    for col in df_solvpn_to_all.columns:
        if col == x_col:
            continue

        fig.add_trace(
            go.Scatter(
                x=df_solvpn_to_all[x_col],
                y=df_solvpn_to_all[col],
                mode="lines",
                name=col.replace(f"{SOLVPN_NAME}_to_", "")
            )
        )

    fig.update_layout(
        title=title,
        xaxis_title=x_col,
        yaxis_title="Directional Spillover (%)",
        height=600,
        hovermode="x unified"
    )

    fig.show()


# ============================================================
# 1. Load GFEVD
# ============================================================

if not GFEVD_FILE.exists():
    raise FileNotFoundError(f"Not found: {GFEVD_FILE.resolve()}")

raw_gfevd = np.load(GFEVD_FILE)
gfevd_tnn = infer_gfevd_tnn(raw_gfevd)

T_eff, N, N2 = gfevd_tnn.shape

if N != N2:
    raise ValueError(f"GFEVD last two dims must be square. Got {gfevd_tnn.shape}")

VAR_NAMES = infer_var_names(N)

print(f"[INFO] Raw GFEVD shape: {raw_gfevd.shape}")
print(f"[INFO] Using GFEVD shape: {gfevd_tnn.shape}")
print(f"[INFO] Variables: {VAR_NAMES}")


# ============================================================
# 2. Normalize GFEVD to percent
# ============================================================

theta_pct_all = normalize_gfevd_to_percent(gfevd_tnn)

print("[INFO] GFEVD row-normalized to percent scale.")
print(f"[INFO] Max row-sum error: {np.max(np.abs(theta_pct_all.sum(axis=2) - 100.0)):.12f}")


# ============================================================
# 3. Load dates
# ============================================================

dates = load_dates(T_eff)

if dates is not None:
    x_col = "Date"
    df_time = pd.DataFrame({"Date": dates})
else:
    x_col = "t"
    df_time = pd.DataFrame({"t": np.arange(T_eff)})

print(f"[INFO] Time index: {x_col}")


# ============================================================
# 4. Compute pairwise net
# ============================================================

pairwise_all = np.zeros((T_eff, N, N), dtype=float)

for t in range(T_eff):
    pairwise_all[t] = compute_pairwise_net(theta_pct_all[t])

pairwise_mean = pairwise_all.mean(axis=0)

df_pairwise_mean = pd.DataFrame(
    pairwise_mean,
    index=VAR_NAMES,
    columns=VAR_NAMES
)

# Wide time-series format
for i, src in enumerate(VAR_NAMES):
    for j, dst in enumerate(VAR_NAMES):
        if i == j:
            continue

        col_name = f"{src}_to_{dst}"
        df_time[col_name] = pairwise_all[:, i, j]


# ============================================================
# 5. Long format
# ============================================================

records = []

for t in range(T_eff):
    time_value = dates.iloc[t] if dates is not None else t

    for i, src in enumerate(VAR_NAMES):
        for j, dst in enumerate(VAR_NAMES):
            if i == j:
                continue

            value = pairwise_all[t, i, j]

            if value > 0:
                direction = f"{src} -> {dst}"
            elif value < 0:
                direction = f"{dst} -> {src}"
            else:
                direction = "Neutral"

            records.append({
                x_col: time_value,
                "Source": src,
                "Target": dst,
                "Pairwise_NET": value,
                "Direction_Interpretation": direction
            })

df_long = pd.DataFrame(records)


# ============================================================
# 6. SOLVPN -> all variables raw directional spillover
# ============================================================

df_solvpn_to_all = None

if SOLVPN_NAME in VAR_NAMES:
    solvpn_idx = VAR_NAMES.index(SOLVPN_NAME)

    df_solvpn_to_all = df_time[[x_col]].copy()

    for target_idx, target_name in enumerate(VAR_NAMES):
        if target_idx == solvpn_idx:
            continue

        # theta[target, source]
        # source = SOLVPN, target = target variable
        col_name = f"{SOLVPN_NAME}_to_{target_name}"
        df_solvpn_to_all[col_name] = theta_pct_all[:, target_idx, solvpn_idx]

else:
    print(f"[WARN] {SOLVPN_NAME} not found. SOLVPN outgoing spillover skipped.")


# ============================================================
# 7. Save CSV / NPY
# ============================================================

OUT_PAIRWISE_ALL = OUT_DIR / "pairwise_net_all.npy"
OUT_PAIRWISE_MEAN = OUT_DIR / "pairwise_net_mean.csv"
OUT_PAIRWISE_LONG = OUT_DIR / "pairwise_net_long.csv"
OUT_PAIRWISE_TIME = OUT_DIR / "pairwise_net_time_wide.csv"

np.save(OUT_PAIRWISE_ALL, pairwise_all)
df_pairwise_mean.to_csv(OUT_PAIRWISE_MEAN, encoding="utf-8-sig")
df_long.to_csv(OUT_PAIRWISE_LONG, index=False, encoding="utf-8-sig")
df_time.to_csv(OUT_PAIRWISE_TIME, index=False, encoding="utf-8-sig")

print("[INFO] Saved pairwise outputs:")
print(f"  - {OUT_PAIRWISE_ALL}")
print(f"  - {OUT_PAIRWISE_MEAN}")
print(f"  - {OUT_PAIRWISE_LONG}")
print(f"  - {OUT_PAIRWISE_TIME}")

if df_solvpn_to_all is not None:
    OUT_SOLVPN_TO_ALL_CSV = OUT_DIR / "solvpn_to_all_spillover.csv"
    df_solvpn_to_all.to_csv(OUT_SOLVPN_TO_ALL_CSV, index=False, encoding="utf-8-sig")

    print("[INFO] Saved SOLVPN outgoing spillover CSV:")
    print(f"  - {OUT_SOLVPN_TO_ALL_CSV}")


# ============================================================
# 8. Save heatmap PNG only
# ============================================================

OUT_HEATMAP = OUT_DIR / "pairwise_net_heatmap.png"
save_heatmap_png(df_pairwise_mean, OUT_HEATMAP)

print("[INFO] Saved heatmap PNG:")
print(f"  - {OUT_HEATMAP}")


# ============================================================
# 9. Save selected pair PNG series
# ============================================================

if SAVE_SELECTED_PAIR_PNG:
    print("[INFO] Saving selected pair PNG series...")

    for src, dst in SELECTED_PAIRS:
        if src not in VAR_NAMES or dst not in VAR_NAMES or src == dst:
            print(f"[WARN] Skip invalid pair: {src} -> {dst}")
            continue

        pair_col = f"{src}_to_{dst}"

        if pair_col not in df_time.columns:
            print(f"[WARN] Pair column not found: {pair_col}")
            continue

        out_png = OUT_DIR / f"{pair_col}.png"
        save_pair_series_png(df_time, x_col, pair_col, out_png)

        print(f"  - {out_png}")


# ============================================================
# 10. Save SOLVPN -> all static PNG
# ============================================================

if df_solvpn_to_all is not None:
    OUT_SOLVPN_TO_ALL_PNG = OUT_DIR / "solvpn_to_all_spillover.png"

    x = pd.to_datetime(df_solvpn_to_all[x_col]) if x_col == "Date" else df_solvpn_to_all[x_col]

    plt.figure(figsize=(14, 6))

    for col in df_solvpn_to_all.columns:
        if col == x_col:
            continue

        plt.plot(
            x,
            df_solvpn_to_all[col],
            linewidth=1.2,
            label=col.replace(f"{SOLVPN_NAME}_to_", "")
        )

    plt.title("SOLVPN → Other Variables Spillover")
    plt.xlabel(x_col)
    plt.ylabel("Directional Spillover (%)")
    plt.legend(ncol=3, fontsize=8)
    plt.tight_layout()
    plt.savefig(OUT_SOLVPN_TO_ALL_PNG, dpi=300, bbox_inches="tight")
    plt.close()

    print("[INFO] Saved SOLVPN outgoing spillover PNG:")
    print(f"  - {OUT_SOLVPN_TO_ALL_PNG}")


# ============================================================
# 11. Interactive line charts inside notebook / Colab only
# ============================================================

if SHOW_INTERACTIVE:
    valid_pair_cols = []

    for src, dst in SELECTED_PAIRS:
        pair_col = f"{src}_to_{dst}"
        if pair_col in df_time.columns:
            valid_pair_cols.append(pair_col)

    if valid_pair_cols:
        show_pair_series_interactive(
            df_time=df_time,
            x_col=x_col,
            pair_cols=valid_pair_cols,
            title="Interactive Selected Pairwise Net Spillover"
        )

    if df_solvpn_to_all is not None:
        show_solvpn_to_all_interactive(
            df_solvpn_to_all=df_solvpn_to_all,
            x_col=x_col,
            title="Interactive SOLVPN → Other Variables Raw Directional Spillover"
        )


# ============================================================
# 12. Focus summary: SOLVPN -> COPPER
# ============================================================

focus_src = "dlog_SOLVPN"
focus_dst = "dlog_COPPER"

if focus_src in VAR_NAMES and focus_dst in VAR_NAMES:
    focus_col = f"{focus_src}_to_{focus_dst}"

    if focus_col in df_time.columns:
        focus_summary = {
            "Pair": focus_col,
            "Mean": df_time[focus_col].mean(),
            "Median": df_time[focus_col].median(),
            "Std": df_time[focus_col].std(),
            "Positive_Ratio": (df_time[focus_col] > 0).mean(),
            "Q10": df_time[focus_col].quantile(0.10),
            "Q25": df_time[focus_col].quantile(0.25),
            "Q75": df_time[focus_col].quantile(0.75),
            "Q90": df_time[focus_col].quantile(0.90),
        }

        df_focus_summary = pd.DataFrame([focus_summary])

        OUT_FOCUS_SUMMARY = OUT_DIR / "solvpn_copper_pairwise_summary.csv"
        df_focus_summary.to_csv(OUT_FOCUS_SUMMARY, index=False, encoding="utf-8-sig")

        print("\n[INFO] SOLVPN -> COPPER pairwise net summary")
        print(df_focus_summary.to_string(index=False))
        print(f"  - {OUT_FOCUS_SUMMARY}")

else:
    print("[WARN] SOLVPN-COPPER focus summary skipped.")

[INFO] Raw GFEVD shape: (1031, 9, 9)
[INFO] Using GFEVD shape: (1031, 9, 9)
[INFO] Variables: ['dlog_SOLVPN', 'dlog_COPPER', 'dlog_GOLD', 'dlog_SILVER', 'dlog_DXY', 'dlog_OIL', 'dlog_GAS', 'd_UST10Y', 'd_VIX']
[INFO] GFEVD row-normalized to percent scale.
[INFO] Max row-sum error: 0.000000000000
[INFO] Time index: t
[INFO] Saved pairwise outputs:
  - pairwise_output\pairwise_net_all.npy
  - pairwise_output\pairwise_net_mean.csv
  - pairwise_output\pairwise_net_long.csv
  - pairwise_output\pairwise_net_time_wide.csv
[INFO] Saved SOLVPN outgoing spillover CSV:
  - pairwise_output\solvpn_to_all_spillover.csv
[INFO] Saved heatmap PNG:
  - pairwise_output\pairwise_net_heatmap.png
[INFO] Saving selected pair PNG series...
  - pairwise_output\dlog_SOLVPN_to_dlog_COPPER.png
  - pairwise_output\dlog_COPPER_to_dlog_SOLVPN.png
  - pairwise_output\dlog_SOLVPN_to_dlog_GOLD.png
  - pairwise_output\dlog_SOLVPN_to_dlog_SILVER.png
  - pairwise_output\dlog_SOLVPN_to_d_VIX.png
[INFO] Saved SOLVPN outgoin


[INFO] SOLVPN -> COPPER pairwise net summary
                      Pair     Mean   Median      Std  Positive_Ratio       Q10       Q25      Q75      Q90
dlog_SOLVPN_to_dlog_COPPER 0.718497 0.757024 5.385503        0.589719 -4.914615 -1.997729 2.869364 5.892127
  - pairwise_output\solvpn_copper_pairwise_summary.csv
